In [1]:
import pandas as pd
from tabulate import tabulate


# Defining a helper function to print any DataFrame as a clean left aligned table
# Using tabulate with tablefmt plain so there are no borders just clean spacing
# Using showindex False to hide the default row numbers on the left
def print_table(df):
    print(tabulate(df, headers=df.columns, tablefmt='plain', showindex=False))


# Loading all three sheets from the Excel file into separate DataFrames
# Using sheet_name to specify which sheet to load from the same Excel file
# Storing each sheet separately because they represent different business entities
xl          = pd.ExcelFile('data/recruitment_interview_data.xlsx')
applicants  = pd.read_excel(xl, sheet_name='applicants')
interviews  = pd.read_excel(xl, sheet_name='interviews')
departments = pd.read_excel(xl, sheet_name='departments')


# Inspecting the structure, columns and shapes of all datasets

print()
print('inspecting applicants table')
print(f'  shape   : {applicants.shape}')
print(f'  columns : {applicants.columns.tolist()}')
print()
print_table(applicants)
print()

print('inspecting interviews table')
print(f'  shape   : {interviews.shape}')
print(f'  columns : {interviews.columns.tolist()}')
print()
print_table(interviews)
print()

print('inspecting departments table')
print(f'  shape   : {departments.shape}')
print(f'  columns : {departments.columns.tolist()}')
print()
print_table(departments)
print()


# Identifying the common key columns for combining the tables

# applicant_id is the common key between applicants and interviews tables
# department_id is the common key between applicants and departments tables
print('identifying common key columns')
print('  applicants  <-> interviews  : applicant_id')
print('  applicants  <-> departments : department_id')
print()


# Checking whether any key columns contain duplicated values

# Checking applicant_id in applicants table
app_dups = applicants['applicant_id'].duplicated().sum()

# Checking applicant_id in interviews table
# Duplicates here are expected because one applicant can have multiple interview stages
int_dups = interviews['applicant_id'].duplicated().sum()

# Checking department_id in departments table
dep_dups = departments['department_id'].duplicated().sum()

print('checking for duplicate key values')
print(f'  applicant_id duplicates in applicants  : {app_dups}')
print(f'  applicant_id duplicates in interviews  : {int_dups}')
print(f'  department_id duplicates in departments: {dep_dups}')
print()
print('  explaining duplicate effect on merge')
print('  applicant 802 appears twice in interviews because they completed')
print('  two interview stages which is expected in a real hiring process')
print('  this causes a one to many relationship and creates extra rows in the merged table')
print()


# Merging the datasets into one final business table

# Merging applicants with interviews using outer join on applicant_id
# Using outer join so we keep all records from both tables even if they do not match
merged = applicants.merge(interviews, on='applicant_id', how='outer')

# Merging the result with departments using left join on department_id
# Using left join so we keep all applicant records and attach department info where available
merged = merged.merge(departments, on='department_id', how='left')

print('merging all datasets into one final business table')
print(f'  final merged shape : {merged.shape}')
print()
print_table(merged)
print()


# Detecting unmatched records across the tables

# Finding applicants who have no matching record in the interviews table
unmatched_apps = applicants[~applicants['applicant_id'].isin(interviews['applicant_id'])]

# Finding interviews that have no matching applicant record in the applicants table
unmatched_int = interviews[~interviews['applicant_id'].isin(applicants['applicant_id'])]

print('detecting unmatched records')
print(f'  applicants with no interview record : {len(unmatched_apps)}')
print()
print_table(unmatched_apps)
print()
print(f'  interviews with no applicant record : {len(unmatched_int)}')
print()
print_table(unmatched_int)
print()


# Identifying missing values in the final merged dataset

# Counting missing values per column to understand data quality issues
missing = merged.isnull().sum().reset_index()
missing.columns = ['column', 'missing_count']

# Filtering to show only columns that actually have missing values
missing = missing[missing['missing_count'] > 0]

print('identifying missing values in the merged dataset')
print()
print_table(missing)
print()


# Creating a new column classifying interview scores into Strong, Average and Weak

# Classifying scores as Strong if 80 or above, Average if between 60 and 79
# and Weak if below 60, using Unknown for rows where score is missing
def classify_score(score):
    if pd.isna(score):  return 'Unknown'
    if score >= 80:     return 'Strong'
    elif score >= 60:   return 'Average'
    else:               return 'Weak'

# Applying the classification function to every row in the score column
merged['score_category'] = merged['score'].apply(classify_score)

# Showing score category distribution as a clean summary table
score_cat = merged['score_category'].value_counts().reset_index()
score_cat.columns = ['score_category', 'count']

print('classifying interview scores into Strong, Average and Weak')
print()
print_table(score_cat)
print()


# Creating a pivot table showing the average interview score by department

# Grouping by department name and calculating the mean score for each department
# Rounding to 2 decimal places for cleaner output
pivot_dept = merged.groupby('department_name')['score'].mean().round(2).reset_index()
pivot_dept.columns = ['department', 'average_score']

print('average interview score by department')
print()
print_table(pivot_dept)
print()


# Creating a summary showing the number of applicants by application status

# Grouping by application status and counting applicant ids in each group
status_summary = merged.groupby('application_status')['applicant_id'].count().reset_index()
status_summary.columns = ['application_status', 'count']

print('number of applicants by application status')
print()
print_table(status_summary)
print()


# Reshaping the data to improve comparison of interview stages across departments

# Using pivot_table to reshape data with departments as rows and interview stages as columns
# This makes it easy to compare how each department performs at each interview stage
# Using mean as the aggregation function and rounding to 2 decimal places
stage_pivot = merged.pivot_table(
    index='department_name',
    columns='interview_stage',
    values='score',
    aggfunc='mean'
).round(2)

# Resetting column name label added by pivot_table for cleaner display
stage_pivot.columns.name = None
stage_pivot = stage_pivot.reset_index()
stage_pivot.columns.name = None

print('reshaping data to compare average scores by department and interview stage')
print()
print_table(stage_pivot)
print()


# Writing a short interpretation of the business insights from the final results

print('interpretation')
print('''
  1. the merged dataset contains 9 rows after combining all three tables showing
     that one applicant had two interview stages causing an extra row in the merge.

  2. one applicant gina has no interview record and one interview record belongs
     to an unknown applicant id 808 indicating data entry gaps that need to be fixed.

  3. finance department has the highest average interview score of 88.0 suggesting
     it is attracting stronger candidates or running a more rigorous evaluation process.

  4. out of all applicants 3 are shortlisted, 3 are in review, 1 is hired and 1 is rejected
     showing the recruitment pipeline is active but conversion to hire is still very low.

  5. score classification shows 4 strong and 4 average candidates with no weak scorers
     meaning the initial screening process is effectively filtering out poor candidates.

  6. the reshaped stage comparison shows that technical interviews in finance produce
     the highest scores while phone interviews in recruitment produce the lowest scores.
''')


inspecting applicants table
  shape   : (7, 4)
  columns : ['applicant_id', 'applicant_name', 'city', 'department_id']

  applicant_id  applicant_name    city     department_id
           801  Ariana            Toronto  HR1
           802  Benjamin          Ajax     HR2
           803  Celine            Oshawa   HR1
           804  Darius            Markham  HR3
           805  Elina             Vaughan  HR2
           806  Farid             Toronto  HR4
           807  Gina              Oshawa   HR3

inspecting interviews table
  shape   : (8, 5)
  columns : ['interview_id', 'applicant_id', 'interview_stage', 'score', 'application_status']

  interview_id    applicant_id  interview_stage      score  application_status
         16001             801  Phone                   78  In Review
         16002             802  Technical               88  Shortlisted
         16003             803  Phone                   69  Rejected
         16004             804  HR                      74 